# 00.- Librerias

In [13]:
import mysql.connector
from mysql.connector import Error
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# 01.- Cargar la Base de datos

In [14]:
def conectar_bd (name_bd:str, host_bd:str, user_bd:str, psw_bd)->tuple:
    """Intenta conectarse a una base de datos MySQL y devuelve los objetos de conexión y cursor asociados.

    Args:
        name_bd (str): Nombre de la BD a conectarse
        host_bn (ste): Nombre del host a conectarse
        user_bd (str): Nombre del usuario que se va a conectar
        psw_bd (str): Password de usuario que se va a conectar

    Returns:
        tuple: Una tupla (connection, cursor). Si la conexión falla, devuelve (None, None).
    """
    try:
        connection = mysql.connector.connect(host = host_bd,
                                        database = name_bd,
                                        user = user_bd,
                                        password = psw_bd)
        if connection.is_connected():
            db_Info = connection.get_server_info()
            print("Connected to MySQL Server version ", db_Info)
            cursor = connection.cursor()
            cursor.execute("select database();")
            record = cursor.fetchone()
            print("You're connected to database: ", record)
            
            return connection, cursor

    except Error as e:
        print("Error while connecting to MySQL", e)        
        return None, None

In [15]:
def tablas_bd (cursor)->pd.DataFrame:
    """Obtiene los nombres de todas las tablas de la base de datos activa y los devuelve en un DataFrame de pandas.

    Args:
        cursor (_type_): Objeto cursor de MySQL utilizado para ejecutar la consulta

    Returns:
        pd.DataFrame: DataFrame con una columna llamada 'tabla' que contiene
        los nombres de todas las tablas. Si ocurre un error, devuelve un
        DataFrame vacío.
    """
    try:
        sql_select_Query = "SHOW TABLES"
        cursor.execute(sql_select_Query)
        records = cursor.fetchall()
        return pd.DataFrame(records, columns=["tabla"])
    
    except Error as e:
        print("Error al obtener el nombre de todas las tablas:", e) 
        return pd.DataFrame() # Devuelve DF vacío si falla
    

In [16]:
def carga_tabla_en_df(cursor, name_tabla:str)->pd.DataFrame:
    """Carga todos los registros de una tabla de la base de datos en un DataFrame de pandas.

    Args:
        cursor (_type_): Objeto cursor de MySQL utilizado para ejecutar la consulta.
        name_tabla (str): Nombre de la tabla que se desea cargar

    Returns:
        pd.DataFrame: DataFrame con los datos de la tabla. 
        Si ocurre un error durante la ejecución de la consulta, se devuelve un DataFrame vacío.
    """
    
    try:
        sql_select_Query = f"select * from {name_tabla}"
        cursor.execute(sql_select_Query)
        records = cursor.fetchall()
        columnas = cursor.column_names        
        return pd.DataFrame(records, columns=columnas)
    
    except Error as e:
        print(f"Error al cargar la tabla {name_tabla} en DF:", e) 
        return pd.DataFrame() # Devuelve DF vacío si falla

In [17]:
def desconectar_bd(connection, cursor):
    """Cierra de forma segura el cursor y la conexión a la base de datos, siempre que ambos estén abiertos.

    Args:
        connection (_type_): Objeto de conexión MySQL
        cursor (_type_): Objeto cursor asociado a la conexión
    """
    try:
        if connection is not None and connection.is_connected():
            cursor.close()
            connection.close()
            print("MySQL connection is closed")
    except Error as e:
        print("Error while closing connection:", e)

    

In [18]:
load_dotenv()

host = os.getenv("HOST")
database = os.getenv("NAME")
user = os.getenv("USUARIO")
psw = os.getenv("PSW")

# print(f"host = {host}")
# print(f"database = {database}")
# print(f"user = {user}")
# print(f"psw = {psw}")


# 1 Conectarme a la BD
conexion, cursor = conectar_bd(database, host, user, psw)

# 2 Saber el nombre de las tablas de la BD
if cursor is None:
    print("No se pudo conectar a la BD. Revisa credenciales o permisos.")
else:
    df_tablas = tablas_bd(cursor)
    display(df_tablas)


# 3 Cargar UNA de las tablas anteriores
nombre_tabla = df_tablas.iloc[0,0]
df_pisos = carga_tabla_en_df(cursor, nombre_tabla)
display(df_pisos)


# # 4.0 Cargara todas las tablas en un diccionario
# # 4.1 Diccionario donde guardaremos todos los DataFrames 
# dic_dfs = {}

# # 4.2 Cargar todas las tabla en un DF cada una
# for nombre in df_tablas["tabla"]:
#     nom = "df_" + nombre
#     # print(nom)
#     nom_valor = carga_tabla_en_df(cursor, nombre)
#     dic_dfs[nom] = nom_valor
#     # display(nom)

# 5 desconectarme de la BD
desconectar_bd(conexion, cursor)

Connected to MySQL Server version  8.0.46
You're connected to database:  ('Equip_29',)


,tabla
0,Tourist_Accommodation


,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,None,Private room,2,2,1,...,100.0,100.0,100.0,100.0,100.0,FALSO,75.0,spain,malaga,31/07/2018
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1,1,...,90.0,100.0,100.0,80.0,90.0,FALSO,52.0,spain,madrid,10/01/2020
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1,2,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,142.0,spain,sevilla,29/07/2019
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2,1,...,90.0,100.0,100.0,100.0,90.0,VERDADERO,306.0,spain,barcelona,10/01/2020
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,None,Private room,5,1,2,...,100.0,100.0,100.0,100.0,100.0,FALSO,39.0,spain,girona,19/02/2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,47710189,Beautiful house in the centre in Santa Cristin...,Beautiful house in the center of Santa Cristin...,263841355,Santa Cristina d'Aro,None,Entire home/apt,8,2,4,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,31/01/2021
9996,47752964,Costa Brava - Palafrugell - playa y monta�a,Piso con acceso directo a piscina. Lugar muy t...,282214688,Palafrugell,None,Entire home/apt,7,2,3,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,girona,27/02/2021
9997,47792016,MODERN AND BRIGHT NEW FLAT IN THE CENTER OF PA...,Modern and new apartment in the old town of Pa...,263841355,Palam�s,None,Entire home/apt,6,1,3,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,31/01/2021
9998,47884481,MIT House Olavide III in Madrid,The apartment is in a characteristic building ...,377605855,Trafalgar,Chamber�,Entire home/apt,4,1,1,...,NaN,NaN,NaN,NaN,NaN,FALSO,NaN,spain,madrid,09/02/2021


MySQL connection is closed


# 02.- Exportar a parquet

In [19]:
df_pisos.to_parquet("2026_06_08_pisos_turisticos_bruto.parquet", index=True)

# 03.- Control de registros

In [20]:
# 1. TOTAL REGISTROS
total_registros = len(df_pisos)

# 2. ID ÚNICOS
id_unicos = df_pisos["apartment_id"].nunique()

# 3. ID EXTRAS (duplicados de ID menos 1)
id_extras = df_pisos["apartment_id"].duplicated().sum()

# 4. ID DUPLICADOS EXACTOS (cuenta todas las columnas iguales. Sin keep=False, cuenta SOLO las repeticiones, NO la primera aparición)
duplicados_exactos = df_pisos.duplicated(keep=False).sum()

# Mostrar resultados
print("CONTROL DE REGISTROS")
print("---------------------")
print(f"TOTAL REGISTROS: {total_registros}")
print(f"ID ÚNICOS: {id_unicos}")
print(f"ID EXTRAS: {id_extras}")
print(f"ID DUPLICADOS EXACTOS: {duplicados_exactos}")



CONTROL DE REGISTROS
---------------------
TOTAL REGISTROS: 10000
ID ÚNICOS: 9650
ID EXTRAS: 350
ID DUPLICADOS EXACTOS: 0


In [21]:
# Tipos de datos e información general
df_pisos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 10000 non-null  int64  
 1   name                         9997 non-null   object 
 2   description                  9862 non-null   object 
 3   host_id                      10000 non-null  int64  
 4   neighbourhood_name           10000 non-null  object 
 5   neighbourhood_district       6079 non-null   object 
 6   room_type                    10000 non-null  object 
 7   accommodates                 10000 non-null  int64  
 8   bathrooms                    9926 non-null   object 
 9   bedrooms                     9930 non-null   object 
 10  beds                         9955 non-null   float64
 11  amenities_list               9983 non-null   object 
 12  price                        9746 non-null   float64
 13  minimum_nights   

# 04.- Duplicados. Identificar y tratar

In [22]:
# Saber cuantos apartment_id estan repetidos 1 vez, 2 veces, 3 veces, ... n veces
# Los que se repiten 1 vez son unicos 
# Los que se repiten mas de 1 vez hay que tratarlos
df_pisos["apartment_id"].value_counts().value_counts()


count
1    9308
2     334
3       8
Name: count, dtype: int64

Comprobar que para cada apartment_id repetido, tiene un insert_date diferente y si es asi, deberemos quedarnos con el registro de fecha menos antigua

In [23]:
# Ver en cada apartment_id su insert_date
df_pisos_duplicados = df_pisos[df_pisos["apartment_id"].duplicated(keep=False)][["apartment_id", "insert_date"]].sort_values("apartment_id")
df_pisos_duplicados.head(10)


,apartment_id,insert_date
22,144471,12/09/2017
23,144471,10/10/2018
24,157327,30/04/2020
25,157327,30/08/2018
50,343864,05/06/2017
51,343864,07/11/2018
89,503253,18/04/2018
90,503253,28/01/2019
225,886569,25/06/2020
224,886569,11/02/2021


In [24]:
# Comprobamos que no hay ninguna fecha de insert_date repetida en un mismos apartment_id
ids_con_fechas_repetidas = (
    df_pisos.groupby("apartment_id")["insert_date"].nunique()
    .lt(
        df_pisos.groupby("apartment_id")["insert_date"].count()
        )
)

# Obtengo los indices de los apartment_id que tienen un insert_date repetido (Trues del less than)
ids_con_fechas_repetidas = ids_con_fechas_repetidas[ids_con_fechas_repetidas].index


# IDs duplicados totales
ids_duplicados = df_pisos["apartment_id"].value_counts()
ids_duplicados = ids_duplicados[ids_duplicados > 1].index

total_ids_duplicados = len(ids_duplicados)
total_ids_con_fechas_repetidas = len(ids_con_fechas_repetidas)

print(f"Total apartment_id duplicados: {total_ids_duplicados}")
print(f"De ellos, IDs con insert_date repetido: {total_ids_con_fechas_repetidas}")


Total apartment_id duplicados: 342
De ellos, IDs con insert_date repetido: 0


In [25]:
# Comprobar el tipo de variable que es insert_date
df_pisos["insert_date"].info()


<class 'pandas.core.series.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: insert_date
Non-Null Count  Dtype 
--------------  ----- 
10000 non-null  object
dtypes: object(1)
memory usage: 78.3+ KB


In [26]:
# Como insert_date es del tipo bbject y para quedarnos con la fecha menos antigua primero, tengo que convetirta a tipo datetime
df_pisos['insert_date'] = pd.to_datetime(df_pisos['insert_date'], format='%d/%m/%Y')

# Comprobar el tipo de variable que es insert_date
df_pisos["insert_date"].info()


<class 'pandas.core.series.Series'>
RangeIndex: 10000 entries, 0 to 9999
Series name: insert_date
Non-Null Count  Dtype         
--------------  -----         
10000 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 78.3 KB


In [27]:
# Ordeno el df_pisos por apartment_id y por insert_date descendente para que se quede en la priemra posición el mas actual de cada uno
df_pisos_ordenado = df_pisos.sort_values(by=['apartment_id', 'insert_date'], ascending=[True, False])

# creo un df nuevo resultado de eliminar los duplicados de 'apartment_id', quedandome solo con el primero (el más actual al ordenarlo)
df_pisos_sin_duplicados = df_pisos_ordenado.drop_duplicates(subset=['apartment_id'], keep='first').copy()

# Reseteo el índice del DataFrame resultante para tenerlo estandarizado
df_pisos_sin_duplicados = df_pisos_sin_duplicados.reset_index(drop=True)

# Vuelvo a comprobar, en el df sin duplicados, cuantos apartment_id estan repetidos 1 vez, 2 veces, 3 veces, ... n veces
df_pisos_sin_duplicados["apartment_id"].value_counts().value_counts()



count
1    9650
Name: count, dtype: int64

# 05 Estandarización de formatos

In [28]:
df_pisos_sin_duplicados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 9650 non-null   int64         
 1   name                         9647 non-null   object        
 2   description                  9516 non-null   object        
 3   host_id                      9650 non-null   int64         
 4   neighbourhood_name           9650 non-null   object        
 5   neighbourhood_district       5860 non-null   object        
 6   room_type                    9650 non-null   object        
 7   accommodates                 9650 non-null   int64         
 8   bathrooms                    9578 non-null   object        
 9   bedrooms                     9580 non-null   object        
 10  beds                         9605 non-null   float64       
 11  amenities_list               9634 non-null 

Transformar Variables temporales

In [29]:
# first_review_date y last_review_date de object a datatime
df_pisos_sin_duplicados['first_review_date'] = pd.to_datetime(df_pisos_sin_duplicados['first_review_date'], format='%d/%m/%Y')
df_pisos_sin_duplicados['last_review_date'] = pd.to_datetime(df_pisos_sin_duplicados['first_review_date'], format='%d/%m/%Y')

df_pisos_sin_duplicados[["first_review_date", "last_review_date"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   first_review_date  7128 non-null   datetime64[ns]
 1   last_review_date   7128 non-null   datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 150.9 KB


Transformar Variables Numéricas

In [30]:
# Convierto a numerico las columnas bathrooms y bedrooms, convertiendo a NaN los errores
df_pisos_sin_duplicados['bathrooms'] = pd.to_numeric(df_pisos_sin_duplicados['bathrooms'], errors='coerce')
df_pisos_sin_duplicados['bedrooms'] = pd.to_numeric(df_pisos_sin_duplicados['bedrooms'], errors='coerce')

df_pisos_sin_duplicados[["bathrooms", "bedrooms"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9650 entries, 0 to 9649
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   bathrooms  9578 non-null   float64
 1   bedrooms   9580 non-null   float64
dtypes: float64(2)
memory usage: 150.9 KB


Transformar variables Booleam

In [31]:
# Ver valores diferentes y frecuencia de cada uno ANTES de la transformacio
print("ANTES TRASFORMACION: ")
display(df_pisos_sin_duplicados["has_availability"].value_counts(dropna=False))

# Transformar VERDADERO a True
df_pisos_sin_duplicados['has_availability'] = df_pisos_sin_duplicados['has_availability'].replace('VERDADERO', True)

# Asumo que los valores nulls seran False por no tener disponibilidad inmediata
# df_pisos_sin_duplicados['has_availability'].fillna(False, inplace = True)
df_pisos_sin_duplicados['has_availability'] = (df_pisos_sin_duplicados['has_availability'].fillna(False))


# Ver valores diferentes y frecuencia de cada uno DESPUES de la transformacio
print("\nDESPUES TRASFORMACION: ")
display(df_pisos_sin_duplicados["has_availability"].value_counts(dropna=False))

ANTES TRASFORMACION: 


has_availability
VERDADERO    9116
None          534
Name: count, dtype: int64


DESPUES TRASFORMACION: 


has_availability
True     9116
False     534
Name: count, dtype: int64

In [32]:
# Ver valores diferentes y frecuencia de cada uno ANTES de la transformacio
print("ANTES TRASFORMACION: ")
display(df_pisos_sin_duplicados["is_instant_bookable"].value_counts(dropna=False))

# Transformar VERDADERO a True
df_pisos_sin_duplicados['is_instant_bookable'] = df_pisos_sin_duplicados['is_instant_bookable'].replace({"VERDADERO" : True, "FALSO" : False}).infer_objects()

# # Asumo que los valores nulls seran False por no tener disponibilidad inmediata
df_pisos_sin_duplicados['is_instant_bookable'] = df_pisos_sin_duplicados['is_instant_bookable'].replace('FALSO', False)

# Ver valores diferentes y frecuencia de cada uno DESPUES de la transformacio
print("\nDESPUES TRASFORMACION: ")
display(df_pisos_sin_duplicados["is_instant_bookable"].value_counts(dropna=False))

ANTES TRASFORMACION: 


is_instant_bookable
VERDADERO    5596
FALSO        4054
Name: count, dtype: int64


DESPUES TRASFORMACION: 


is_instant_bookable
True     5596
False    4054
Name: count, dtype: int64

Variables de ratings: Tienen un intervalo maximo de 100 y 1000 cuando en la documentacion dice que es 100 para la variable "review_scores_rating" y 10 para el resto

Como no sabemos como se llega al "review_scores_rating" partiendo del resto de variables, voy a dividirla entre 10 para que coincida su maximo con el resto de variables

In [33]:
columnas = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

print("Min y MAX ANTES de normalizar:")
# display(df_pisos_sin_duplicados[columnas].agg(['min', 'max']))
display(df_pisos_sin_duplicados[columnas].describe())


# Dividir entre 10 para ponerla en la misma escala que en las otras
df_pisos_sin_duplicados['review_scores_rating'] = (df_pisos_sin_duplicados['review_scores_rating'] / 10)


print("Min y MAX DESPUES de normalizar:")
# display(df_pisos_sin_duplicados[columnas].agg(['min', 'max']))
display(df_pisos_sin_duplicados[columnas].describe())

Min y MAX ANTES de normalizar:


,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,7025.000000,7016.000000,7022.000000,7011.000000,7020.000000,7010.000000,7010.000000
mean,918.967972,94.492588,93.105953,96.220225,96.200855,95.436519,91.350927
std,92.966960,9.503675,10.129198,8.325030,8.353365,7.695836,10.008605
min,200.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
25%,890.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000
50%,940.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.000000
75%,980.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
max,1000.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


Min y MAX DESPUES de normalizar:


,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,7025.000000,7016.000000,7022.000000,7011.000000,7020.000000,7010.000000,7010.000000
mean,91.896797,94.492588,93.105953,96.220225,96.200855,95.436519,91.350927
std,9.296696,9.503675,10.129198,8.325030,8.353365,7.695836,10.008605
min,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
25%,89.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000
50%,94.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.000000
75%,98.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


# 06.- Exportar df limpio a falta de Nulls

In [34]:
df_pisos_sin_duplicados.to_parquet("2026_06_08_pisos_turisticos_sin_duplicados.parquet", index=True)

# 07.- Nulls (NaN + vacíos + espacios). Identificar

## 07.1.- Nulos

In [35]:
# Ver cuántos NULOS hay en cada columna
df_pisos_sin_duplicados.isna().sum()


apartment_id                      0
name                              3
description                     134
host_id                           0
neighbourhood_name                0
neighbourhood_district         3790
room_type                         0
accommodates                      0
bathrooms                        72
bedrooms                         70
beds                             45
amenities_list                   16
price                           241
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              2522
last_review_date               2522
review_scores_rating           2625
review_scores_accuracy         2634
review_scores_cleanliness      2628
review_scores_checkin          2639
review_scores_communication 

Bathrooms: Les asignamos la mediana de baños por persona segun la capacidad

**Decisión de imputación de bathrooms:** Usamos la mediana de baños por nivel
de `accommodates` porque hay correlación lógica (más capacidad → más baños).


In [36]:
# Numero de bathrooms NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['bathrooms'].isna().sum()}")


# Calcula la mediana de 'bathrooms' por cada valor de 'accommodates'
mediana_bathrooms_por_capacidad = df_pisos_sin_duplicados.groupby('accommodates')['bathrooms'].median()

# Asigno esa mediana a los valores NaN
def asignar_bathrooms(row):
    if pd.isnull(row['bathrooms']):
        return mediana_bathrooms_por_capacidad.get(row['accommodates'], df_pisos_sin_duplicados['bathrooms'].median()) # Si no hay mediana para ese 'accommodates', devuelve la mediana general
    return row['bathrooms']

df_pisos_sin_duplicados['bathrooms'] = df_pisos_sin_duplicados.apply(asignar_bathrooms, axis=1)

# Numero de bathrooms NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['bathrooms'].isna().sum()}")

'Nº de NaN ANTES = 72'

'Nº de NaN DESPUES = 0'

Bedrooms: Les asignamos la mediana de dormitorios por persona segun la capacidad

**Decisión de imputación de Bedrooms:** Usamos la mediana de Bedrooms por nivel
de `accommodates` porque hay correlación lógica (más capacidad → más Bedrooms).

In [37]:
# Numero de bedrooms NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['bedrooms'].isna().sum()}")


# Calcula la mediana de 'bedrooms' por cada valor de 'accommodates'
mediana_bedrooms_por_capacidad = df_pisos_sin_duplicados.groupby('accommodates')['bedrooms'].median()

# Asigno esa mediana a los valores NaN
def asignar_bedrooms(row):
    if pd.isnull(row['bedrooms']):
        return mediana_bedrooms_por_capacidad.get(row['accommodates'], df_pisos_sin_duplicados['bedrooms'].median()) # Si no hay mediana para ese 'accommodates', devuelve la mediana general
    return row['bedrooms']

df_pisos_sin_duplicados['bedrooms'] = df_pisos_sin_duplicados.apply(asignar_bathrooms, axis=1)

# Numero de bedrooms NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['bedrooms'].isna().sum()}")

'Nº de NaN ANTES = 70'

'Nº de NaN DESPUES = 0'

Beds: Les asignamos la mediana de camas por persona segun la capacidad

**Decisión de imputación de beds:** Usamos la mediana de beds por nivel
de `accommodates` porque hay correlación lógica (más capacidad → más beds).

In [38]:
# Numero de beds NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['beds'].isna().sum()}")


# Calcula la mediana de 'beds' por cada valor de 'accommodates'
mediana_beds_por_capacidad = df_pisos_sin_duplicados.groupby('accommodates')['beds'].median()

# Asigno esa mediana a los valores NaN
def asignar_bathrooms(row):
    if pd.isnull(row['beds']):
        return mediana_beds_por_capacidad.get(row['accommodates'], df_pisos_sin_duplicados['beds'].median()) # Si no hay mediana para ese 'accommodates', devuelve la mediana general
    return row['beds']

df_pisos_sin_duplicados['beds'] = df_pisos_sin_duplicados.apply(asignar_bathrooms, axis=1)

# Numero de beds NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['beds'].isna().sum()}")

'Nº de NaN ANTES = 45'

'Nº de NaN DESPUES = 0'

Price: Les asignaremos la mediana del precio que tengan en la misma ciudad el mismo tipo de alojamiento

In [39]:
# Numero de beds NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['price'].isna().sum()}")

# Calculo la mediana de 'price' por 'city' y 'room_type'
mediana_price_por_city_roomtype = df_pisos_sin_duplicados.groupby(['city', 'room_type'])['price'].median()

# Función para imputar los valores nulos en 'price'
def asignar_price(row):
    if pd.isnull(row['price']):
        try:
            return mediana_price_por_city_roomtype[(row['city'], row['room_type'])]
        except KeyError:
            return df_pisos_sin_duplicados['price'].median() # Si no existe la combinación, usa la mediana general
    return row['price']

# Aplicar la imputación
df_pisos_sin_duplicados['price'] = df_pisos_sin_duplicados.apply(asignar_price, axis=1)


# Numero de beds NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['price'].isna().sum()}")


'Nº de NaN ANTES = 241'

'Nº de NaN DESPUES = 0'

In [40]:
# Ver cuántos NULOS hay en cada columna DESPUES de la gestión
df_pisos_sin_duplicados.isna().sum()


apartment_id                      0
name                              3
description                     134
host_id                           0
neighbourhood_name                0
neighbourhood_district         3790
room_type                         0
accommodates                      0
bathrooms                         0
bedrooms                          0
beds                              0
amenities_list                   16
price                             0
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              2522
last_review_date               2522
review_scores_rating           2625
review_scores_accuracy         2634
review_scores_cleanliness      2628
review_scores_checkin          2639
review_scores_communication 

## 07.2.- Cadena vacia ""

In [41]:
# Detectar cadenas vacías ""
(df_pisos_sin_duplicados == "").sum()


apartment_id                   0
name                           0
description                    0
host_id                        0
neighbourhood_name             0
neighbourhood_district         0
room_type                      0
accommodates                   0
bathrooms                      0
bedrooms                       0
beds                           0
amenities_list                 0
price                          0
minimum_nights                 0
maximum_nights                 0
has_availability               0
availability_30                0
availability_60                0
availability_90                0
availability_365               0
number_of_reviews              0
first_review_date              0
last_review_date               0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
is_instant

## 07.3.- Cadena con espacios

In [42]:
# # Detectar cadenas con espacios " " (vacíos “disfrazados”)
df_pisos_sin_duplicados.apply(
    lambda col: col.astype(str).str.strip().eq("").sum()
    if col.dtype == "object" else 0
)


apartment_id                   0
name                           0
description                    0
host_id                        0
neighbourhood_name             0
neighbourhood_district         0
room_type                      0
accommodates                   0
bathrooms                      0
bedrooms                       0
beds                           0
amenities_list                 0
price                          0
minimum_nights                 0
maximum_nights                 0
has_availability               0
availability_30                0
availability_60                0
availability_90                0
availability_365               0
number_of_reviews              0
first_review_date              0
last_review_date               0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
is_instant

In [43]:
# Detectar cadenas con espacios " " (vacíos “disfrazados”)
# df_pisos_sin_duplicados.apply(lambda col: col.str.strip().eq("").sum() if col.dtype == "object" else 0)


## 07.4.- Todo junto (NaN + vacíos + espacios)

In [44]:
# Detectar todo junto (NaN + vacíos + espacios)
df_pisos_sin_duplicados.apply(
    lambda col: (
        col.isna() | 
        (col.astype(str).str.strip() == "")
    ).sum()
)


apartment_id                      0
name                              3
description                     134
host_id                           0
neighbourhood_name                0
neighbourhood_district         3790
room_type                         0
accommodates                      0
bathrooms                         0
bedrooms                          0
beds                              0
amenities_list                   16
price                             0
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              2522
last_review_date               2522
review_scores_rating           2625
review_scores_accuracy         2634
review_scores_cleanliness      2628
review_scores_checkin          2639
review_scores_communication 

# 09.- Comprobaciones finales

In [45]:
# Ver si los valores oara el indice que usamos apartment_id son unicos
if (df_pisos_sin_duplicados['apartment_id'].is_unique):
    print("Todos los valores de 'apartment_id' son unicos\n")
else:
    print("Hay valores de 'apartment_id' duplicados\n")
    
# Ver si todos los valores review_scores_rating estan dentro del rango 0-100
if (df_pisos_sin_duplicados['review_scores_rating'].dropna().between(0, 100).all()):
    print("Todos los valores de 'review_scores_rating' estan entre 0 y 100\n")
else:
    print("Hay valores de 'review_scores_rating' estan fuera del rango entre 0 y 100\n")

# Ver los registros o filas finales que nos quedan despues de la limpieza
print(f"Filas finales: {len(df_pisos_sin_duplicados)}\n")
print(f"Columnas: {df_pisos_sin_duplicados.shape[1]}\n")

# Existen NaN en algún campo de algun registro???
if (df_pisos_sin_duplicados.isna().any().any()):
    print("Existen valores NaN en alguna columna\n")
else:
    print("NO xisten valores NaN en ninguna columna\n")

# % de valores NaN por columna para ver si estan en las variables que vamos a tratar
print(f"% NaN por columna:\n{(df_pisos_sin_duplicados.isna().mean()*100).round(2)}")

Todos los valores de 'apartment_id' son unicos

Todos los valores de 'review_scores_rating' estan entre 0 y 100

Filas finales: 9650

Columnas: 35

Existen valores NaN en alguna columna

% NaN por columna:
apartment_id                    0.00
name                            0.03
description                     1.39
host_id                         0.00
neighbourhood_name              0.00
neighbourhood_district         39.27
room_type                       0.00
accommodates                    0.00
bathrooms                       0.00
bedrooms                        0.00
beds                            0.00
amenities_list                  0.17
price                           0.00
minimum_nights                  0.00
maximum_nights                  0.00
has_availability                0.00
availability_30                 0.00
availability_60                 0.00
availability_90                 0.00
availability_365                0.00
number_of_reviews               0.00
first_review_date

In [46]:
# Cuantos tipos de pisos hay en cada ciudad
df_pisos_sin_duplicados.groupby(["city", "room_type"]).size()


city       room_type      
barcelona  Entire home/apt    1237
           Hotel room           22
           Private room       1430
           Shared room          30
girona     Entire home/apt    1390
           Hotel room           11
           Private room         83
           Shared room           1
madrid     Entire home/apt    1294
           Hotel room           14
           Private room        796
           Shared room          30
malaga     Entire home/apt     405
           Hotel room            4
           Private room         92
           Shared room           3
mallorca   Entire home/apt    1415
           Hotel room           15
           Private room        149
           Shared room           4
menorca    Entire home/apt     198
           Private room         20
sevilla    Entire home/apt     385
           Hotel room           11
           Private room         97
           Shared room           1
valencia   Entire home/apt     353
           Hotel room       

# 09.- Exportar df limpio final

In [47]:
df_pisos_sin_duplicados.to_parquet("2026_06_08_pisos_turisticos_limpio.parquet", index=True)